In [ ]:
# Jupyter Notebook: 03_full_pipeline.ipynb
# Full Pipeline Demonstration

# Cell 1: Imports
import numpy as np
import matplotlib.pyplot as plt
from experiments.synthetic_data import SyntheticDriftGenerator
from src.wavelets.dwt_pipeline import DWTDecomposer
from src.ensemble.heterogeneous import MultiResolutionEnsemble
from utils.metrics import DriftMetrics
from utils.visualization import DriftVisualizer

print("="*60)
print("Full Wavelet Drift Detection Pipeline Demonstration")
print("="*60)

# Cell 2: Generate Synthetic Data
print("\n1. Generating synthetic data...")
gen = SyntheticDriftGenerator(seed=42)
X, y, true_drifts = gen.generate(drift_type='sudden', n=5000)
print(f"✓ Generated stream with drift at {true_drifts}")

# Cell 3: Initialize DWT Decomposer
print("\n2. Initializing DWT decomposer...")
decomposer = DWTDecomposer(wavelet='db4', level=4)
print(f"✓ Using wavelet: db4, decomposition level: 4")

# Decompose first sample
decomp_sample = decomposer.decompose(y[:100])
print(f"✓ Decomposition has {len(decomp_sample)} components")

# Cell 4: Compute Wavelet Energies
print("\n3. Computing wavelet energies...")
n_window = 200
energies_hist = {}
energies_new = {}

# Historical window
hist_decomp = decomposer.decompose(y[:n_window])
for j in hist_decomp.keys():
    energies_hist[j] = decomposer.compute_energy(hist_decomp[j])

# New window (at true drift time)
new_decomp = decomposer.decompose(y[true_drifts[0]:true_drifts[0]+n_window])
for j in new_decomp.keys():
    energies_new[j] = decomposer.compute_energy(new_decomp[j])

print("Wavelet Energies:")
print("Scale | Historical | New Window | Change")
print("-" * 45)
for j in sorted(energies_hist.keys()):
    change = (energies_new[j] - energies_hist[j]) / (energies_hist[j] + 1e-10)
    print(f"{j:5d} | {energies_hist[j]:10.2f} | {energies_new[j]:10.2f} | {change:+.2%}")

# Cell 5: Visualize Wavelet Energies
print("\n4. Visualizing wavelet energies...")

scales = sorted(energies_hist.keys())
hist_vals = [energies_hist[j] for j in scales]
new_vals = [energies_new[j] for j in scales]

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(scales))
width = 0.35
ax.bar(x - width/2, hist_vals, width, label='Historical', alpha=0.7)
ax.bar(x + width/2, new_vals, width, label='New Window', alpha=0.7)
ax.set_xlabel('Scale')
ax.set_ylabel('Energy')
ax.set_title('Wavelet Energy Comparison at Drift Point')
ax.set_xticks(x)
ax.set_xticklabels([f'Scale {j}' for j in scales])
ax.legend()
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

# Cell 6: Initialize Ensemble
print("\n5. Initializing multi-resolution ensemble...")

# Prepare multi-scale training data
n_init = 500
X_init_dict = {}
y_init_dict = {}

for j in range(5):  # 5 scales
    # For now, use same X for all scales
    X_init_dict[j] = X[:n_init]
    
    # Decompose y and extract scale j components
    y_decomp = decomposer.decompose(y[:n_init])
    y_init_dict[j] = y_decomp[j][:n_init]

ensemble = MultiResolutionEnsemble(J=4)
ensemble.fit(X_init_dict, y_init_dict, val_size=100)

print("✓ Ensemble trained")
print(f"✓ Initial weights: {ensemble.initial_weights}")

# Cell 7: Make Predictions
print("\n6. Making predictions...")

predictions = []
errors = []

for t in range(n_init, min(n_init + 500, len(y))):
    X_t_dict = {j: X[t:t+1] for j in range(5)}
    
    y_pred = ensemble.predict(X_t_dict)
    y_true = y[t]
    
    predictions.append(y_pred)
    errors.append(np.abs(y_true - y_pred))

print(f"✓ Generated {len(predictions)} predictions")
print(f"✓ Mean absolute error: {np.mean(errors):.4f}")

# Cell 8: Visualize Predictions
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Predictions vs True
axes[0].plot(y[n_init:n_init+len(predictions)], 'b-', alpha=0.7, label='True', linewidth=1)
axes[0].plot(predictions, 'r-', alpha=0.7, label='Predicted', linewidth=1)
axes[0].set_ylabel('Y')
axes[0].set_title('Predictions vs True Values')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Errors
axes[1].plot(errors, 'g-', alpha=0.7, linewidth=0.5)
axes[1].axhline(np.mean(errors), color='r', linestyle='--', label=f'Mean={np.mean(errors):.4f}')
axes[1].set_xlabel('Time')
axes[1].set_ylabel('Absolute Error')
axes[1].set_title('Prediction Errors')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Cell 9: Summary
print("\n" + "="*60)
print("Pipeline Summary")
print("="*60)
print(f"Total stream length: {len(y)}")
print(f"Warm-up size: {n_init}")
print(f"Evaluation samples: {len(predictions)}")
print(f"Mean absolute error: {np.mean(errors):.4f}")
print(f"Predictions range: [{np.min(predictions):.3f}, {np.max(predictions):.3f}]")
print("="*60)
print("✓ Full pipeline execution complete!")